# Barça History RAG Assistant — Pipeline Notebook

This notebook builds and evaluates the full RAG pipeline: load → chunk → embed → store → retrieve → generate → evaluate.

**Domain:** FC Barcelona history, signings, and season-by-season stats.
**Sources:** FC Barcelona official "Decade by Decade" history pages, Wikipedia (History of FC Barcelona, List of FC Barcelona seasons, Camp Nou).
**Track:** Core (text-only RAG).

## 2.1 Load & Inspect

In [1]:
import json
from pathlib import Path

PROCESSED_DIR = Path('../data/processed')
manifest = json.loads((PROCESSED_DIR / '_manifest.json').read_text(encoding='utf-8'))

docs = []
for entry in manifest['documents']:
    # NOTE: the manifest stores paths relative to the project root (where
    # clean_text.py was run from), but this notebook's working directory is
    # notebooks/. So we take just the filename and join it with PROCESSED_DIR,
    # which we already resolved correctly above.
    filename = Path(entry['processed_file']).name
    path = PROCESSED_DIR / filename
    text = path.read_text(encoding='utf-8')
    docs.append({
        'source_group': entry['source_group'],
        'filename': path.name,
        'char_count': entry['char_count'],
        'text': text,
    })

print(f"Total documents loaded: {len(docs)}")
print(f"Documents failed during scraping/cleaning: {manifest['documents_failed']}")
print()
for d in docs:
    print(f"[{d['source_group']:12}] {d['filename']:55} {d['char_count']:>7} chars")

Total documents loaded: 16
Documents failed during scraping/cleaning: 0

[fcbarcelona ] fcbarcelona__1899-1909-foundation-and-survival.txt         3942 chars
[fcbarcelona ] fcbarcelona__1909-19-consolidation-at-carrer-industria.txt    3817 chars
[fcbarcelona ] fcbarcelona__1919-30-a-golden-age.txt                      3620 chars
[fcbarcelona ] fcbarcelona__1930-39-struggling-against-history.txt        4822 chars
[fcbarcelona ] fcbarcelona__1939-50-years-of-perseverance.txt             4160 chars
[fcbarcelona ] fcbarcelona__1950-1961-the-kubala-era.txt                  4041 chars
[fcbarcelona ] fcbarcelona__1961-69-a-new-social-dimension.txt            3741 chars
[fcbarcelona ] fcbarcelona__1969-78-cruyff-and-democracy.txt              4911 chars
[fcbarcelona ] fcbarcelona__1978-88-more-members-more-stars.txt           4785 chars
[fcbarcelona ] fcbarcelona__1988-1996-the-era-of-the-dream-team.txt       4513 chars
[fcbarcelona ] fcbarcelona__1996-2008-barca-reaches-its-century.txt       

**How many documents/pages?** 16 processed text documents: 13 from the FC Barcelona official "Decade by Decade" history section (one per decade, 1899–2021), and 3 from Wikipedia (`History_of_FC_Barcelona`, `List_of_FC_Barcelona_seasons`, `Camp_Nou`).

**What formats?** All 16 are plain UTF-8 `.txt` files. They started as live HTML pages, scraped with `requests`/`trafilatura` and cleaned in `scraper/clean_text.py` — no PDFs or scanned documents were used, so there was no OCR step needed.

**Which files failed to parse or need OCR?** None. All 16 source pages were text-extractable HTML; `clean_text.py` reported `0 failed` (its `MIN_USABLE_CHARS` threshold catches any page trafilatura fails to extract meaningfully, and nothing tripped it).

## 2.2 Chunking Strategy

In [2]:
import re

CHUNK_SIZE = 800       # target characters per chunk
CHUNK_OVERLAP = 150    # characters of overlap between consecutive chunks

SENTENCE_SPLIT_RE = re.compile(r'(?<=[.!?])\s+')

def split_into_paragraphs(text: str) -> list[str]:
    # trafilatura's extracted text separates blocks with single newlines
    # (not blank lines), so split on any newline rather than \n\s*\n.
    paras = [p.strip() for p in text.split('\n') if p.strip()]
    return paras

def split_oversized(paragraph: str, chunk_size: int) -> list[str]:
    """Safety net: if a single paragraph is itself longer than chunk_size
    (e.g. a dense Wikipedia prose block), break it into sentences so the
    packer below can still respect the chunk size budget."""
    if len(paragraph) <= chunk_size:
        return [paragraph]
    sentences = SENTENCE_SPLIT_RE.split(paragraph)
    return [s.strip() for s in sentences if s.strip()]

def chunk_document(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Paragraph-aware chunking: greedily pack paragraphs up to chunk_size,
    then carry the tail of the previous chunk forward as overlap so retrieval
    doesn't lose context that straddles a chunk boundary."""
    raw_paragraphs = split_into_paragraphs(text)
    paragraphs = []
    for p in raw_paragraphs:
        paragraphs.extend(split_oversized(p, chunk_size))
    chunks = []
    current = ''
    for para in paragraphs:
        candidate = (current + '\n\n' + para).strip() if current else para
        if len(candidate) <= chunk_size:
            current = candidate
        else:
            if current:
                chunks.append(current)
            # start new chunk, carrying overlap from the end of the previous one
            tail = current[-overlap:] if current else ''
            current = (tail + '\n\n' + para).strip() if tail else para
    if current:
        chunks.append(current)
    return chunks

all_chunks = []  # list of dicts: {text, source_group, filename, chunk_index}
for d in docs:
    doc_chunks = chunk_document(d['text'])
    for i, c in enumerate(doc_chunks):
        all_chunks.append({
            'text': c,
            'source_group': d['source_group'],
            'filename': d['filename'],
            'chunk_index': i,
        })

print(f"Total chunks created: {len(all_chunks)}")
print(f"Average chunk size: {sum(len(c['text']) for c in all_chunks) / len(all_chunks):.0f} chars")
print(f"Chunks per document (avg): {len(all_chunks) / len(docs):.1f}")

Total chunks created: 340
Average chunk size: 672 chars
Chunks per document (avg): 21.2


**Why 800 chars / 150 overlap, and paragraph-aware rather than fixed-size?**

The source material is narrative prose organized in paragraphs (decade summaries, Wikipedia prose, season tables rendered as text) rather than dense technical text, so splitting mid-sentence with a naive fixed-size window risks cutting a signing, date, or score in half. Packing whole paragraphs up to a ~800-character budget keeps each chunk topically coherent (roughly one Barça "beat" — a signing, a trophy, a managerial change — per chunk) while staying well within the embedding model's context comfortably.

150 characters of overlap (carried from the tail of the previous chunk) protects against the common failure mode where a key fact (e.g. a transfer year) sits right at a paragraph boundary and would otherwise only be retrievable from one side of the split.

**Implementation note:** the paragraph splitter looks for newline-separated blocks (matching how `trafilatura` formats extracted text) rather than blank-line-separated blocks, and any single block that still exceeds `chunk_size` is further broken into sentences as a fallback — otherwise a single dense paragraph (common in the Wikipedia sources) would become one oversized, unfocused chunk.

## 2.3 Embeddings & Vector Store

In [3]:
from sentence_transformers import SentenceTransformer
import chromadb

EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'
VECTOR_STORE_DIR = '../data/vector_store'
COLLECTION_NAME = 'barca_history'

embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

chunk_texts = [c['text'] for c in all_chunks]
print(f'Embedding {len(chunk_texts)} chunks with {EMBEDDING_MODEL_NAME}...')
embeddings = embedder.encode(chunk_texts, show_progress_bar=True, convert_to_numpy=True)
print(f'Embedding shape: {embeddings.shape}')

Embedding 340 chunks with all-MiniLM-L6-v2...


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Embedding shape: (340, 384)


In [4]:
client = chromadb.PersistentClient(path=VECTOR_STORE_DIR)

# fresh collection each run of this notebook
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass
collection = client.create_collection(COLLECTION_NAME, metadata={'hnsw:space': 'cosine'})

ids = [f"{c['filename']}::chunk{c['chunk_index']}" for c in all_chunks]
metadatas = [
    {'source_group': c['source_group'], 'filename': c['filename'], 'chunk_index': c['chunk_index']}
    for c in all_chunks
]

collection.add(
    ids=ids,
    embeddings=embeddings.tolist(),
    documents=chunk_texts,
    metadatas=metadatas,
)

print(f"Persisted {collection.count()} chunks to Chroma at '{VECTOR_STORE_DIR}' (collection: '{COLLECTION_NAME}')")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Persisted 340 chunks to Chroma at '../data/vector_store' (collection: 'barca_history')


## 2.4 Retrieval & Prompting

In [5]:
def retrieve(query: str, k: int = 4) -> list[dict]:
    """Embed the query and return the top-k most similar chunks with metadata."""
    query_embedding = embedder.encode([query], convert_to_numpy=True).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=k)
    hits = []
    for text, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
        hits.append({'text': text, 'metadata': meta, 'distance': dist})
    return hits

def build_prompt(question: str, hits: list[dict]) -> str:
    """Combine retrieved context with the question, with explicit citation instructions."""
    context_blocks = []
    for i, h in enumerate(hits, start=1):
        source = f"{h['metadata']['source_group']}/{h['metadata']['filename']}"
        context_blocks.append(f"[Source {i}: {source}]\n{h['text']}")
    context = '\n\n'.join(context_blocks)

    return f"""You are a knowledgeable FC Barcelona history assistant. Answer the question ONLY using the context below. \
If the context does not contain the answer, say you don't have enough information — do not use outside knowledge.
After your answer, list which Source numbers you used.

Context:
{context}

Question: {question}

Answer:"""

# quick sanity check on one query before the full evaluation
sample_hits = retrieve('When did Johan Cruyff sign for Barcelona?')
for h in sample_hits:
    print(f"- {h['metadata']['filename']} (chunk {h['metadata']['chunk_index']}, distance={h['distance']:.3f})")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


- wikipedia__history_of_fc_barcelona.txt (chunk 24, distance=0.259)
- fcbarcelona__1969-78-cruyff-and-democracy.txt (chunk 2, distance=0.339)
- wikipedia__history_of_fc_barcelona.txt (chunk 25, distance=0.373)
- wikipedia__history_of_fc_barcelona.txt (chunk 34, distance=0.389)


In [6]:
import ollama

OLLAMA_MODEL = 'llama3.2:3b'

def generate_answer(question: str, k: int = 4) -> dict:
    hits = retrieve(question, k=k)
    prompt = build_prompt(question, hits)
    response = ollama.generate(model=OLLAMA_MODEL, prompt=prompt)
    sources = sorted({h['metadata']['filename'] for h in hits})
    return {'question': question, 'answer': response['response'].strip(), 'sources': sources, 'hits': hits}

TEST_QUESTIONS = [
    'When was FC Barcelona founded and by whom?',
    'When did Johan Cruyff sign for Barcelona as a player?',
    'What happened to the club during and after the Spanish Civil War?',
    'What was the Dream Team era and who managed it?',
    'When did Barcelona win their first European Cup?',
    'What is Camp Nou and when did it open?',
    'What role did Ronaldinho play in the 2000s?',
    'What major trophies did Barcelona win in the 2008-2011 period?',
    'How did the club celebrate its centenary?',
    'What changes happened to Barcelona around 2021?',
]

results = [generate_answer(q) for q in TEST_QUESTIONS]
for r in results:
    print(f"Q: {r['question']}")
    print(f"A: {r['answer'][:300]}{'...' if len(r['answer']) > 300 else ''}")
    print(f"Sources: {r['sources']}")
    print()

Q: When was FC Barcelona founded and by whom?
A: FC Barcelona was founded in 1899 by a group of Swiss, Catalan, German, and English footballers led by Joan Gamper.

Source numbers used:

1. Source 1: wikipedia/wikipedia__history_of_fc_barcelona.txt
2. Source 3: wikipedia/wikipedia__list_of_fc_barcelona_seasons.txt
Sources: ['fcbarcelona__1899-1909-foundation-and-survival.txt', 'fcbarcelona__1930-39-struggling-against-history.txt', 'wikipedia__history_of_fc_barcelona.txt', 'wikipedia__list_of_fc_barcelona_seasons.txt']

Q: When did Johan Cruyff sign for Barcelona as a player?
A: According to Source 2: fcbarcelona/fcbarcelona__1969-78-cruyff-and-democracy.txt, Johan Cruyff signed for Barcelona as a player on 13 August 1973.

Source numbers used:

1. fcbarcelona/fcbarcelona__1969-78-cruyff-and-democracy.txt
2. wikipedia/wikipedia__history_of_fc_barcelona.txt
Sources: ['fcbarcelona__1969-78-cruyff-and-democracy.txt', 'wikipedia__history_of_fc_barcelona.txt']

Q: What happened to the club d

## 2.6 Evaluation

In [7]:
import pandas as pd

# Judgments below are based on manually checking each answer against known
# FC Barcelona history and against the sources it cited.
MANUAL_JUDGMENTS = {
    'When was FC Barcelona founded and by whom?': True,
    'When did Johan Cruyff sign for Barcelona as a player?': True,
    'What happened to the club during and after the Spanish Civil War?': True,
    'What was the Dream Team era and who managed it?': True,
    'When did Barcelona win their first European Cup?': False,  # retrieval miss, see analysis below
    'What is Camp Nou and when did it open?': True,
    'What role did Ronaldinho play in the 2000s?': True,
    'What major trophies did Barcelona win in the 2008-2011 period?': True,
    'How did the club celebrate its centenary?': True,
    'What changes happened to Barcelona around 2021?': True,
}

eval_rows = []
for r in results:
    eval_rows.append({
        'question': r['question'],
        'retrieved_sources': ', '.join(r['sources']),
        'answer': r['answer'][:200] + ('...' if len(r['answer']) > 200 else ''),
        'correct': MANUAL_JUDGMENTS.get(r['question']),
    })

eval_df = pd.DataFrame(eval_rows)
accuracy = eval_df['correct'].mean()
print(f"Accuracy: {eval_df['correct'].sum()}/{len(eval_df)} ({accuracy:.0%})")
eval_df

Accuracy: 9/10 (90%)


,question,retrieved_sources,answer,correct
0,When was FC Barcelona founded and by whom?,fcbarcelona__1899-1909-foundation-and-survival...,FC Barcelona was founded in 1899 by a group of...,True
1,When did Johan Cruyff sign for Barcelona as a ...,"fcbarcelona__1969-78-cruyff-and-democracy.txt,...",According to Source 2: fcbarcelona/fcbarcelona...,True
2,What happened to the club during and after the...,fcbarcelona__1930-39-struggling-against-histor...,"The club experienced terrible times of social,...",True
3,What was the Dream Team era and who managed it?,fcbarcelona__1988-1996-the-era-of-the-dream-te...,The 'Dream Team' era was a period in FC Barcel...,True
4,When did Barcelona win their first European Cup?,"fcbarcelona__1919-30-a-golden-age.txt, fcbarce...",I don't have enough information to answer this...,False
5,What is Camp Nou and when did it open?,"fcbarcelona__1950-1961-the-kubala-era.txt, wik...",Camp Nou is the stadium of FC Barcelona and it...,True
6,What role did Ronaldinho play in the 2000s?,fcbarcelona__2008-20-the-best-years-in-our-his...,Ronaldinho played as a key player in the team ...,True
7,What major trophies did Barcelona win in the 2...,fcbarcelona__2008-20-the-best-years-in-our-his...,Barcelona won the following major trophies in ...,True
8,How did the club celebrate its centenary?,fcbarcelona__1939-50-years-of-perseverance.txt...,The club celebrated its centenary by aiming to...,True
9,What changes happened to Barcelona around 2021?,"wikipedia__camp_nou.txt, wikipedia__history_of...","According to the provided context, around 2021...",True


### Mitigation experiment: retrying the missed question with a higher `k`

In [8]:
# The European Cup question failed at k=4 -- the model correctly said it lacked
# enough information rather than guessing, but the right chunk (1992 Wembley win
# over Sampdoria) apparently wasn't in the top 4 retrieved chunks. Let's check
# whether widening retrieval to k=6 surfaces it.
retry = generate_answer('When did Barcelona win their first European Cup?', k=6)
print('Answer with k=6:')
print(retry['answer'])
print()
print('Sources used:', retry['sources'])

Answer with k=6:
Barcelona won their first European Cup in 1979, specifically on May 16, 1979, when they defeated Fortuna Düsseldorf 4-3 in the Cup Winners’ Cup final.

Source numbers used:

1. [Source 3: fcbarcelona/fcbarcelona__1978-88-more-members-more-stars.txt]
2. [Source 1: fcbarcelona/fcbarcelona__1978-88-more-members-more-stars.txt]

Sources used: ['fcbarcelona__1919-30-a-golden-age.txt', 'fcbarcelona__1939-50-years-of-perseverance.txt', 'fcbarcelona__1978-88-more-members-more-stars.txt', 'fcbarcelona__1988-1996-the-era-of-the-dream-team.txt']


**Failure analysis:** 9 of the 10 test questions were answered correctly and grounded in the retrieved context. The one failure — "When did Barcelona win their first European Cup?" — turned out to be more interesting than a simple retrieval miss.

At `k=4`, the model correctly said it lacked enough information, since the retrieved chunks covered other trophies but not the 1992 European Cup final. The experiment above retried at `k=6` to test whether widening retrieval would fix it — **it did not**. At `k=6`, the retrieved sources did include `fcbarcelona__1988-1996-the-era-of-the-dream-team.txt` (the file that covers the actual 1992 Wembley win over Sampdoria), yet the model answered with **1979 vs Fortuna Düsseldorf — which was the Cup Winners' Cup, a different competition**, not the European Cup. So this wasn't purely a recall problem: the right source document was present in context, but the model conflated two distinct Barça European trophies from the 1970s–90s and answered confidently with the wrong one.

That's a more concerning failure mode than the `k=4` result: going from 'I don't have enough information' to a fluent, specific, wrong answer is exactly the kind of confident hallucination a RAG system is supposed to prevent. The mitigations worth trying next aren't just a bigger `k` — they're (1) chunking with slightly more overlap/context around trophy-name mentions so competition names stay attached to their dates, (2) a stricter prompt instruction telling the model to quote the exact competition name from the source rather than paraphrase it, and (3) a disambiguation check for questions involving named entities that have multiple similarly-described instances (here: multiple European trophies).

## 2.7 Export

In [9]:
import json

config = {
    'embedding_model': EMBEDDING_MODEL_NAME,
    'ollama_model': OLLAMA_MODEL,
    'chunk_size': CHUNK_SIZE,
    'chunk_overlap': CHUNK_OVERLAP,
    'collection_name': COLLECTION_NAME,
    'total_chunks': len(all_chunks),
    'total_documents': len(docs),
}

config_path = f'{VECTOR_STORE_DIR}/config.json'
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)

print(f"Vector store persisted at: {VECTOR_STORE_DIR}")
print(f"Config saved at: {config_path}")
print(json.dumps(config, indent=2))
print()
print('Next: copy this vector_store folder into backend/data/vector_store/ so the FastAPI backend can load it without rebuilding.')

Vector store persisted at: ../data/vector_store
Config saved at: ../data/vector_store/config.json
{
  "embedding_model": "all-MiniLM-L6-v2",
  "ollama_model": "llama3.2:3b",
  "chunk_size": 800,
  "chunk_overlap": 150,
  "collection_name": "barca_history",
  "total_chunks": 340,
  "total_documents": 16
}

Next: copy this vector_store folder into backend/data/vector_store/ so the FastAPI backend can load it without rebuilding.
